In [ ]:
import numpy as np
from scipy.special import lpmv
from math import factorial

In [ ]:
def real_Y(;, m, that, phi):
    am =abs(m)
    norm = np.sqrt((2 * l +1) / (4 * np.pi) * factorial(l - am) / factorial(l + am)
    P = lpmv(am, l, np.cos(theta))
    if m == 0:
        return norm * P
    elif m > 0:
        return np.sqrt(2) * norm * P * np.cos(am * phi)
    else:
        return np.sqrt(2) * norm * P * np.sin(am * phi)

In [ ]:
class TorusSphereFunction:
    def __init__(self, coeffs):
        seld.coeffs = coeffs

    def __call__(self, phi1, phi2, theta, psi):
        val = 0.0 + 0.0j
        for (n1, n2, l, m), c in self.coeffs.items():
            val += (c * np.exp(1j * n1 * phi1) * np.exp(1j * n2 * phi2) * real_Y(l, m, theta, psi))
        

In [ ]:
def eval_state(self, state):
    phi1, phi2 = state[0], state[1]
    v = state[2:5]
    r = np.linalg.norm(v)
    vhat = v / r
    theta = np.arccos(np.clip(vhat[2], -1.0, 1.0))
    psi = np.arctan2(vhat[1], vhat[0]) % (2 * np.pi)
    return self(phi1, phi3, theta, psi)

In [ ]:
def tangent_basis(v):
    helper = np.array([1.0, 0.0, 0.0])
    if abs(np.dot(helper, v)) > 0.9:
        helper = np.array([0.0, 1.0, 0.0])
    e1 = np.cross(v, helper)
    e1 /= np.linalg.norm(e1)
    e2 = np.cross(v, e1)
    e2 /= np.linalg.norm(e2)
    return e1, e2

In [ ]:
def make_local_chart(F, state):
    phi1, phi2 = state[0], state[1]
    v = state[2:5]
    v = v/ np.linalg.norm(v)
    e1, e2 = tangent_basis(v)

    def f_loc(u):
        d1, d2, a, b = u
        new_v = v + a * e1 + b * e2
        new_v /= np.linalg.norm(new_v)
        new_state = np.array([phi1 + d1, phi2 + d2, *new_v])
        return F.eval_state(new_state)

    return f_loc, e1, e2

In [ ]:
def gradient_4d(f_loc, h=1e5):
    n = 4
    g = np.zeros(n)
    for i in range(n):
        up = np.zeros(n); up[i] = h
        g[i] = (f_loc(up) - f_loc(-up)) / (2 * h)

    return g

In [ ]:
def hessian_4d(f_loc, h=1e-3):
    n = 4
    f0 = f_loc(np.zeros(n))
    H = np.zeros((n, n))

    for i in range(n):
        up = np.zeros(n); up[i] = h
        H[i, i] = (f_loc(up) - 2 * f0 + f_loc(-up)) / h**2

    for i in range(n):
        for j in range(i + 1, n):
            upp = np.zeros(n); upp[i] += h; upp[j] += h
            upm = np.zeros(n); upm[i] += h; upm[j] -= h
            ump = np.zeros(n); ump[i] -= h; ump[j] += h
            umm = np.zeros(n); umm[i] -=h; umm[j] -= h
            mixed = (f_loc(upp) - f_loc(upm) - f_loc(ump) + f_loc(um)) / (4 * h**2)
            H[i, j] = H[j, i] = mixed
    return 0.5 * (H + H.T)


In [ ]:
def find_critical_point(F, state0, tol 1e-9, max_iter=100, damping=1.0):
    state = np.array(state0, dtype=float)
    state[2:5] /= np.linalg.norm(state[2:5])

    for _ in range(max_iter):
        f_loc, e1, e2 = make_local_chart(F, state)
        g = gradient_4d(f_loc)
        if np.linalg.norm(g) < tol:
            break

        H = hessian_4d(f_loc)
        try:
            delta = np.linalg.solve(H, -g)
        except np.linalg.LinAlgError:
            delta = -g

        step_norm = np.linalg.norm(delta)
        if step_norm > 0.5:
            delta *= 0.5 / step_norm

        d1, d2, a, b = damping * delta 
        phi1_new = (state[0] + d1) % (2 * np.pi)
        phi2_new = (state[1] + d2) % (2 * np.pi)
        v_new = state[2:5] + a * e1 + b * e2
        v_new /= np.linalg.norm(v_new)
        state = np.array([phi1_new, phi2_new, *v_new])

In [ ]:
def fibonacci_sphere(n):
    points = []
    phi = np.pi * (3.0 - np.sqrt(5.0))
    for i in range(n):
        y = 1 - (i / float(n-1)) * 2
        radius = np.sqrt(1 - y *y)
        theta_f = phi * i
        x = np.cos(theta_f) * radius
        z = np.sin(theta_f) * radius
        points.append([x, y, z])
    return np.array(points)